<a href="https://colab.research.google.com/github/Pserrod869/Optativa/blob/main/practica_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Qué es el análisis de sentimiento**

Es una tarea de clasificación de texto. Dado un texto, el modelo decide si la intención utilizada es positiva, negativa o neutral.

No es "**entender**" el texto como lo haría una persona. Es detectar patrones de palabras y contexto que, estadísticamente, se asocian a cada categoría.

Para esta práctica en español, el modelo más adecuado es **RoBERTuito**. Es un modelo entrenado específicamente con **500 millones de tweets en español**, lo que le permite entender jerga, emojis, abreviaturas y texto informal.

Está diseñado para redes sociales, que es donde el análisis de sentimiento tiene más aplicación real. Y funciona bien en tareas de sentimiento, odio e ironía en español.

RoBERTuito no se entrenó desde cero. Es un modelo preentrenado: alguien invirtió muchísimo tiempo y recursos en entrenarlo con textos masivos para que "entienda" el español.

Luego ese modelo base se ajustó (mediante fine-tuning) con un conjunto más pequeño de tweets etiquetados como positivos, negativos o neutros. El resultado es un modelo especializado en análisis de sentimiento en español, listo para usar.

Esto es importante porque explica por qué tú no entrenas nada. Usas un modelo que ya existe, ya está entrenado, y solo lo llamas desde Colab. Es exactamente lo que haría un desarrollador profesional: **no reinventar la rueda, usar lo que ya funciona**.

# **Hugging Face**

Hugging Face es la plataforma donde viven estos modelos. Su función **pipeline** es una abstracción: tú le dices "quiero realizar un análisis de sentimiento" y "usa este modelo", y ella se encarga de todo lo demás (preprocesar el texto, pasarlo por el modelo, devolver el resultado).

Solo se necesita entender que pipeline es el "botón" que conecta el texto con un modelo ya entrenado.

El modelo devuelve dos cosas:

    Una etiqueta: POS, NEG o NEU

    Una confianza (score), un número entre 0 y 1

La confianza no es lo mismo que **"acierto"**. Es la seguridad del modelo en su propia predicción. Un score de 0.99 no garantiza que tenga razón; un score de 0.51 en una frase ambigua es normal y no es un fallo.

El modelo no lee intenciones, lee patrones. El guiño o el signo de interrogación alteran el contexto que el modelo aprendió durante el preentrenamiento, y eso mueve la predicción.

In [18]:
from transformers import pipeline

clasificador = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis")
# carga un modelo de inteligencia artificial
# entrenado para analizar el sentimiento
# de textos en español, con especial enfoque
# en lenguaje informal y redes sociales

# === este es el texto  que se clasifica ===
texto = "Me encanta este módulo de IA"
# =====================================

resultado = clasificador(texto)
print(resultado)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'POS', 'score': 0.9514580368995667}]


In [19]:
# clasifica un texto que sea negativo

texto = "No me gusta nada esta pelicula, esta muy mal hecha."
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEG', 'score': 0.9825284481048584}]


In [20]:
# clasifica un texto neutro

texto = "El tren sale mañana a las ocho de la mañana desde la estación central."
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEU', 'score': 0.869546115398407}]


In [21]:
# clasifica un texto irónico

texto = "Que bien! Otra reunión que podría haber sido un correo electrónico."
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEU', 'score': 0.639512836933136}]


In [22]:
# clasifica un texto usando emojis

texto = "El concierto de anoche estuvo espectacular! 🎤🎶🔥"
resultado = clasificador(texto)
print(resultado)

[{'label': 'POS', 'score': 0.973301351070404}]


**¿Cómo funciona el análisis de sentimientos? ¿Qué límites tiene?**

### ¿Cómo funciona?
Detecta patrones estadísticos y matemáticos en el texto basándose en millones de datos previos. No entiende el significado real ni tiene conciencia; solo asocia palabras y contextos con probabilidades de ser POS, NEG o NEU.

### Límites principales:
* Sarcasmo e Ironía: Difíciles de detectar al usar palabras positivas con intención negativa.
* Dependencia del Contexto: Una misma palabra cambia de sentido según la situación.
* Ambigüedad: Oraciones complejas o neutras confunden la predicción.
* Falta de comprensión humana: Si el texto es muy diferente a los datos de entrenamiento, el modelo fallará.

# **Práctica**

Implementa en python un analizador de comentarios. Los comentarios positivos y nuetros serán aceptados, salvo que contengan alguna palabra de la lista negra de palabras. Los comentarios negativos que contengan alguna palabra de la lista negra o que tengan un score de más de 0.5 serán rechazadas.

In [23]:
lista_negra = [
    "estafa", "larper", "idiota", "malo", "peor",
    "terrible", "fraude", "tonto", "inútil", "horrible"
]

def clasificar_comentario(comentario):
    resultado_ia = clasificador(comentario)[0]
    sentimiento = resultado_ia['label']
    score = resultado_ia['score']

    comentario_min = comentario.lower()
    contiene_palabra_prohibida = any(palabra in comentario_min for palabra in lista_negra)

    if contiene_palabra_prohibida:
        decision = "RECHAZADO (Contiene palabra prohibida)"
    elif sentimiento == 'NEG':
        decision = f"RECHAZADO (Sentimiento Negativo detectado con confianza: {score:.2f})"
    else:
        decision = "ACEPTADO"

    print(f"Comentario: '{comentario}'")
    print(f"-> Sentimiento: {sentimiento} | Score: {score:.2f} | ¿Palabra Prohibida?: {contiene_palabra_prohibida}")
    print(f"-> DECISIÓN: {decision}\n")

# Lista de 10 frases de prueba para nuestra mini aplicación
frases_prueba = [
    "Me encanta el nuevo diseño de la página web, es fantástico.",
    "El envío llegó un poco tarde, pero el artículo está bien.",
    "Este producto es una total estafa, exijo mi dinero de vuelta.",
    "No me convence para nada, es un dispositivo bastante malo.",
    "¿Alguien sabe a qué hora abre la tienda mañana por la mañana?",
    "La comida estuvo deliciosa y el servicio fue muy atento.",
    "El soporte técnico fue de gran ayuda y solucionó mi problema.",
    "Este tio es un larper total, no se ha visto la peli",
    "No entiendo el funcionamiento del menú de configuración.",
    "La calidad del material es horrible, no lo recomiendo."
]

# Ejecutar la mini aplicación evaluando las 10 frases
print("=== MINI APLICACIÓN DE MODERACIÓN DE COMENTARIOS (10 FRASES) ===\n")
for i, frase in enumerate(frases_prueba, 1):
    print(f"[{i}/10]")
    clasificar_comentario(frase)

=== MINI APLICACIÓN DE MODERACIÓN DE COMENTARIOS (10 FRASES) ===

[1/10]
Comentario: 'Me encanta el nuevo diseño de la página web, es fantástico.'
-> Sentimiento: POS | Score: 0.98 | ¿Palabra Prohibida?: False
-> DECISIÓN: ACEPTADO

[2/10]
Comentario: 'El envío llegó un poco tarde, pero el artículo está bien.'
-> Sentimiento: POS | Score: 0.80 | ¿Palabra Prohibida?: False
-> DECISIÓN: ACEPTADO

[3/10]
Comentario: 'Este producto es una total estafa, exijo mi dinero de vuelta.'
-> Sentimiento: NEG | Score: 0.97 | ¿Palabra Prohibida?: True
-> DECISIÓN: RECHAZADO (Contiene palabra prohibida)

[4/10]
Comentario: 'No me convence para nada, es un dispositivo bastante malo.'
-> Sentimiento: NEG | Score: 0.96 | ¿Palabra Prohibida?: True
-> DECISIÓN: RECHAZADO (Contiene palabra prohibida)

[5/10]
Comentario: '¿Alguien sabe a qué hora abre la tienda mañana por la mañana?'
-> Sentimiento: NEU | Score: 0.87 | ¿Palabra Prohibida?: False
-> DECISIÓN: ACEPTADO

[6/10]
Comentario: 'La comida estuvo del